# Lab 2 — Autocorrelation: what counts as periodicity?

Autocorrelation can be understood literally as **overlap with delayed copies**. We start from binary onset sequences so the geometry is visible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.ndimage import gaussian_filter1d
from IPython.display import Audio, display

np.set_printoptions(precision=3, suppress=True)

def unbiased_autocorrelation(x, max_lag=None, normalize=True):
    x = np.asarray(x, dtype=float).reshape(-1)
    if max_lag is None:
        max_lag = len(x)-1
    max_lag = min(int(max_lag), len(x)-1)
    r = np.empty(max_lag+1)
    for tau in range(max_lag+1):
        a=x[:len(x)-tau]; b=x[tau:]
        r[tau] = np.dot(a,b)/max(1,len(a))
    if normalize and r[0] > 0:
        r = r/r[0]
    return r

def phase_autocorrelation(x, lags, unbiased=True):
    x=np.asarray(x,dtype=float).reshape(-1)
    lags=np.asarray(lags,dtype=int)
    P=np.zeros((len(lags), int(lags.max())))
    C=np.zeros_like(P)
    N=len(x)
    for ir,tau in enumerate(lags):
        for phi in range(tau):
            idx=np.arange(phi, N-tau, tau, dtype=int)
            if len(idx):
                P[ir,phi]=np.sum(x[idx]*x[idx+tau])
                C[ir,phi]=len(idx)
    if unbiased:
        return P/np.where(C==0,1,C), C
    return P, C

def frame_signal(x, frame_length, hop_length, center=True):
    x=np.asarray(x,float).reshape(-1)
    if center:
        p=frame_length//2
        x=np.pad(x,(p,p))
    starts=np.arange(0,max(1,len(x)-frame_length+1),hop_length,dtype=int)
    return np.stack([x[s:s+frame_length] for s in starts],axis=1), starts

def rhythmogram(x, frame_length=256, hop_length=32, lag_min=8, lag_max=96):
    frames, starts = frame_signal(x, frame_length, hop_length, center=True)
    lags=np.arange(lag_min,lag_max+1)
    R=np.zeros((len(lags),frames.shape[1]))
    w=np.hanning(frame_length)
    for j in range(frames.shape[1]):
        f=(frames[:,j]-frames[:,j].mean())*w
        ac=unbiased_autocorrelation(f, lag_max, normalize=False)
        R[:,j]=np.maximum(ac[lags],0)
        m=R[:,j].max()
        if m>0: R[:,j]/=m
    return R,lags,starts

def dp_tempo_path(R, lags, smoothness=30.0):
    R=np.asarray(R,float)
    eps=1e-10
    obs=-np.log(R+eps)
    nP,nT=R.shape
    cost=np.full((nP,nT),np.inf); back=np.zeros((nP,nT),dtype=int)
    cost[:,0]=obs[:,0]
    lp=np.log(np.asarray(lags,float))
    trans=smoothness*(lp[:,None]-lp[None,:])**2
    for t in range(1,nT):
        M=cost[:,t-1][None,:]+trans
        back[:,t]=np.argmin(M,axis=1)
        cost[:,t]=obs[:,t]+M[np.arange(nP),back[:,t]]
    path=np.zeros(nT,dtype=int); path[-1]=np.argmin(cost[:,-1])
    for t in range(nT-2,-1,-1): path[t]=back[path[t+1],t+1]
    return np.asarray(lags)[path]

def simple_novelty(x, fs, n_fft=1024, hop=256):
    f,t,Z=signal.stft(x,fs=fs,window='hann',nperseg=n_fft,noverlap=n_fft-hop,boundary=None,padded=False)
    mag=np.abs(Z)
    flux=np.maximum(mag[:,1:]-mag[:,:-1],0).sum(axis=0)
    flux=np.r_[0,flux]
    if flux.max()>0: flux/=flux.max()
    return flux,t

In [ ]:
N=96
x=np.zeros(N); x[::12]=1
fig,ax=plt.subplots(figsize=(12,2.5)); ax.stem(x); ax.set_title('Periodic onset train: period 12 samples'); ax.set_xlabel('sample'); plt.show()
r=unbiased_autocorrelation(x,48,normalize=False)
plt.figure(figsize=(10,3)); plt.stem(np.arange(len(r)),r); plt.xlabel('lag'); plt.title('Unbiased autocorrelation'); plt.show()

## 1. See the overlap directly

In [ ]:
lags=[6,12,18]
fig,axes=plt.subplots(len(lags),1,figsize=(12,6),sharex=True)
for ax,tau in zip(axes,lags):
    y=np.roll(x,tau); y[:tau]=0
    ax.stem(x,linefmt='C0-',markerfmt='C0o',basefmt=' ')
    ax.stem(y,linefmt='C1-',markerfmt='C1s',basefmt=' ')
    ax.set_title(f'lag {tau}: overlap = {np.dot(x,y):.0f}')
plt.tight_layout(); plt.show()

## 2. Missing and syncopated events

Autocorrelation measures repeated spacing, not whether every expected onset is present.

In [ ]:
variants={
'perfect':x.copy(),
'missing one':x.copy(),
'extra syncopation':x.copy()
}
variants['missing one'][36]=0
variants['extra syncopation'][42]=1
plt.figure(figsize=(11,4))
for name,v in variants.items():
    rr=unbiased_autocorrelation(v,48,normalize=True)
    plt.plot(rr,label=name)
plt.xlabel('lag'); plt.ylabel('normalized autocorrelation'); plt.legend(); plt.title('Periodicity survives imperfect patterns'); plt.show()

## 3. What autocorrelation loses

Translate the whole pattern by five samples. The spacing structure is unchanged, so the autocorrelation is almost unchanged. This is useful for period estimation, but it means **absolute phase has been averaged away**. Lab 3 restores it.